# Questão 2 — Pós-Treino (SFT) com Qwen2.5-0.5B
### Dataset: docentesDC · Pares Instruction/Input/Output
---
> Execute as células **de cima para baixo**.
> A Célula 3 é onde você cola a lista de 1.000+ pares gerados a partir do `docentesDC`.

## Célula 1 — Instalação

In [1]:
!pip install -q transformers>=4.45.0 datasets>=2.20.0 accelerate>=0.34.0 tqdm
print("OK")

OK



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Célula 2 — Imports, Configurações e Carregamento do Modelo Base

In [2]:
import re, math, json, random, copy
import torch
from collections import Counter
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)

# ── Reprodutibilidade ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# ── Configurações ──────────────────────────────────────────────────────────
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "Qwen/Qwen2.5-0.5B"   # modelo
MAX_LEN    = 256
EPOCHS     = 3
LR         = 5e-5

print(f"Dispositivo : {DEVICE}")
print(f"Modelo      : {MODEL_NAME}")

# ── Carrega tokenizer e modelo base ────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
model_base.to(DEVICE)

n_params = sum(p.numel() for p in model_base.parameters())
print(f"Parâmetros  : {n_params/1e6:.1f}M")
print(f"Vocabulário : {tokenizer.vocab_size:,} tokens")

c:\Users\oscar\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo : cuda
Modelo      : Qwen/Qwen2.5-0.5B
Parâmetros  : 494.0M
Vocabulário : 151,643 tokens


## Célula 3 — Pares Instruction/Input/Output (docentesDC)

> **⚠️ COLE AQUI a lista gerada a partir do `docentesDC`.**
> Formato esperado: lista de dicionários com as chaves `instruction`, `input` (opcional, pode ser `""`) e `output`.
> Abaixo está um exemplo mínimo com 5 pares — **substitua pela sua lista com 1.000+ pares**.

In [ ]:
import json, re, random

random.seed(42)

CAMINHO_JSONL = "corpus_treino.jsonl"   # ajuste o caminho se necessário
MIN_CHARS     = 200                      # ignora trechos muito curtos

# ── Carrega os registros do corpus ─────────────────────────────────────────
registros = []
with open(CAMINHO_JSONL, encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if not linha:
            continue
        d = json.loads(linha)
        texto = d.get("text", "").strip()
        if len(texto) < MIN_CHARS:
            continue
        registros.append(d)

print(f"Registros válidos: {len(registros):,}")

# ── Mapeia nomes de tópicos para uma forma legível em pergunta ─────────────
TOPICO_LEGIVEL = {
    "python":                  "Python",
    "banco_dados":              "Banco de Dados",
    "sistemas_operacionais":    "Sistemas Operacionais",
    "estrutura_dados":          "Estrutura de Dados",
    "compiladores":             "Compiladores",
    "algoritmos":               "Algoritmos",
    "redes":                    "Redes de Computadores",
    "programacao_funcional":    "Programação Funcional",
    "eletronica_digital":       "Eletrônica Digital",
}

def topico_valido(t):
    """Verifica se o tópico não é nan/None/vazio."""
    return isinstance(t, str) and t.strip() != ""

def primeiro_trecho_limpo(texto, max_chars=400):
    """Extrai um resumo curto do início do texto, removendo numeração solta."""
    t = re.sub(r'^\s*\d+\s*', '', texto)         # remove número de slide no início
    t = re.sub(r'\s{2,}', ' ', t)                  # normaliza espaços
    t = t.strip()
    if len(t) > max_chars:
        # corta em um ponto final próximo do limite, se existir
        corte = t[:max_chars]
        ultimo_ponto = corte.rfind('.')
        t = corte[:ultimo_ponto+1] if ultimo_ponto > 100 else corte + "..."
    return t

print("Funções auxiliares definidas.")

# ── Templates de pergunta (varia para não repetir a mesma frase sempre) ────
TEMPLATES_COM_TOPICO = [
    "O que o material de {topico} ministrado pelo professor {professor} aborda neste trecho?",
    "Explique o conteúdo sobre {topico} apresentado pelo professor {professor}.",
    "Resuma o que é tratado neste trecho da disciplina de {topico}, do professor {professor}.",
    "Qual conceito de {topico} é discutido no material do professor {professor}?",
]

TEMPLATES_SEM_TOPICO = [
    "O que o professor {professor} aborda neste trecho de material de aula?",
    "Resuma o conteúdo deste trecho do material do professor {professor}.",
    "Explique o que é tratado neste trecho do material do professor {professor}.",
]

def gerar_par(registro):
    """Gera 1 par instruction/input/output a partir de um registro do corpus."""
    texto      = registro["text"]
    professor  = registro.get("nome_professor", "desconhecido").title()
    topico_raw = registro.get("topico_inferido")

    resposta = primeiro_trecho_limpo(texto)

    if topico_valido(topico_raw):
        topico = TOPICO_LEGIVEL.get(topico_raw, topico_raw.replace("_", " ").title())
        pergunta = random.choice(TEMPLATES_COM_TOPICO).format(topico=topico, professor=professor)
    else:
        pergunta = random.choice(TEMPLATES_SEM_TOPICO).format(professor=professor)

    return {
        "instruction": pergunta,
        "input": "",
        "output": resposta
    }

print("Template de geração definido.")

# ── Gera os pares para todos os registros válidos ──────────────────────────
dados_sft = [gerar_par(reg) for reg in registros]

print(f"Total de pares gerados: {len(dados_sft):,}")

if len(dados_sft) < 500:
    print(f"⚠️  Apenas {len(dados_sft)} pares — enunciado pede pelo menos 1.000.")
else:
    print("✓ Quantidade suficiente (>= 1.000 pares exigidos pelo enunciado).")

print("\nExemplo de par gerado:")
print(json.dumps(dados_sft[0], ensure_ascii=False, indent=2))
print()
print(json.dumps(dados_sft[10], ensure_ascii=False, indent=2))

# ── Salva os pares gerados em disco (para reutilizar sem regenerar) ────────
with open("dados_sft_docentesDC.json", "w", encoding="utf-8") as f:
    json.dump(dados_sft, f, ensure_ascii=False, indent=2)

print("Pares salvos em dados_sft_docentesDC.json")

Registros válidos: 3,111
Funções auxiliares definidas.
Template de geração definido.
Total de pares gerados: 3,111
✓ Quantidade suficiente (>= 1.000 pares exigidos pelo enunciado).

Exemplo de par gerado:
{
  "instruction": "Explique o que é tratado neste trecho do material do professor Weslley Emmanuel Martins Lima.",
  "input": "",
  "output": "PonteiroPonteiros são tipos especiais de dados que armazenam endereços de memória.Uma variável do tipo ponteiro deve ser declarada da seguinte forma:tipo *nome_variavel;A variável ponteiro armazenará um endereço de memória de uma outra variável do tipo especificado.Exemplo: int *mema;memaarmazena endereço de memória de variáveis do tipo int."
}

{
  "instruction": "O que o material de Sistemas Operacionais ministrado pelo professor Weslley Emmanuel Martins Lima aborda neste trecho?",
  "input": "",
  "output": "Se a base de computação confiável estiver funcionando de acordo com a especificação, a segurança do sistema não pode ser comprometida,

## Célula 4 — Formatação dos Pares no Padrão de Prompt (Alpaca-style)

In [4]:
def formatar_prompt(exemplo):
    """
    Formata cada par instruction/input/output no estilo Alpaca,
    padrão amplamente usado para SFT de LLMs causais.
    """
    if exemplo.get("input", "").strip():
        prompt = (
            f"### Instrução:\n{exemplo['instruction']}\n\n"
            f"### Entrada:\n{exemplo['input']}\n\n"
            f"### Resposta:\n{exemplo['output']}"
        )
    else:
        prompt = (
            f"### Instrução:\n{exemplo['instruction']}\n\n"
            f"### Resposta:\n{exemplo['output']}"
        )
    return prompt + tokenizer.eos_token

textos_formatados = [formatar_prompt(ex) for ex in dados_sft]

print("=== Exemplo formatado ===")
print(textos_formatados[0])
print(f"\nTotal formatado: {len(textos_formatados):,}")

=== Exemplo formatado ===
### Instrução:
Explique o que é tratado neste trecho do material do professor Weslley Emmanuel Martins Lima.

### Resposta:
PonteiroPonteiros são tipos especiais de dados que armazenam endereços de memória.Uma variável do tipo ponteiro deve ser declarada da seguinte forma:tipo *nome_variavel;A variável ponteiro armazenará um endereço de memória de uma outra variável do tipo especificado.Exemplo: int *mema;memaarmazena endereço de memória de variáveis do tipo int.<|endoftext|>

Total formatado: 3,111


## Célula 5 — Split Treino/Avaliação

In [5]:
random.shuffle(textos_formatados)

N_EVAL = max(50, int(len(textos_formatados) * 0.1))   # 10% para avaliação (mín. 50)

textos_eval   = textos_formatados[:N_EVAL]
textos_treino = textos_formatados[N_EVAL:]

print(f"Treino    : {len(textos_treino):,}")
print(f"Avaliação : {len(textos_eval):,}")

Treino    : 2,800
Avaliação : 311


## Célula 6 — Função de Avaliação Quantitativa (PPL, Entropia, Acurácia)

In [6]:
def calcular_metricas(modelo, textos, batch_size=8):
    """
    Calcula Entropia Cruzada, Perplexidade e Acurácia de previsão de tokens
    — as três métricas pedidas no enunciado.
    """
    modelo.eval()
    total_loss   = 0.0
    total_tokens = 0
    total_certos = 0

    for i in range(0, len(textos), batch_size):
        batch = textos[i : i + batch_size]

        enc = tokenizer(
            batch, return_tensors="pt",
            max_length=MAX_LEN, truncation=True, padding="max_length"
        )
        input_ids      = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100   # ignora padding no loss

        with torch.no_grad():
            out = modelo(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        n_tok = attention_mask.sum().item()
        total_loss   += out.loss.item() * n_tok
        total_tokens += n_tok

        # Acurácia: pred[t] vs token real[t+1]
        preds      = out.logits.argmax(dim=-1)
        mask_shift = attention_mask[:, 1:].bool()
        certos     = (preds[:, :-1] == input_ids[:, 1:])[mask_shift].sum().item()
        total_certos += certos

    ec  = total_loss / total_tokens
    ppl = math.exp(ec)
    acc = total_certos / total_tokens

    return {
        "entropia_cruzada": round(ec, 4),
        "perplexidade":     round(ppl, 2),
        "acuracia_tokens":  round(acc * 100, 2)
    }

print("Função de avaliação definida.")

Função de avaliação definida.


## Célula 7 — Perguntas de Teste (geração qualitativa antes/depois)

In [7]:
# Perguntas de teste para comparar respostas geradas ANTES e DEPOIS do SFT.
# Use perguntas no mesmo estilo dos pares de treino (sobre docentesDC).
perguntas_teste = [
    "Quem é o professor responsável pela disciplina de Tópicos em IA?",
    "Qual é a titulação do professor X do Departamento de Computação?",
    "Em que área de pesquisa atua o professor Y?",
]

def gerar_resposta(modelo, instrucao, max_new_tokens=80):
    modelo.eval()
    prompt = f"### Instrução:\n{instrucao}\n\n### Resposta:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = modelo.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    texto = tokenizer.decode(out[0], skip_special_tokens=True)
    return texto.split("### Resposta:")[-1].strip()

print("Funções de geração definidas.")

Funções de geração definidas.


## Célula 8 — Avaliação ANTES do SFT

In [8]:
print("Calculando métricas do modelo base (sem SFT)...")
metricas_antes = calcular_metricas(model_base, textos_eval)

print("\n=== ANTES do SFT ===")
print(f"  Entropia Cruzada : {metricas_antes['entropia_cruzada']}")
print(f"  Perplexidade     : {metricas_antes['perplexidade']}")
print(f"  Acurácia Tokens  : {metricas_antes['acuracia_tokens']}%")

print("\n--- Respostas ANTES do SFT ---")
for p in perguntas_teste:
    print(f"\nPergunta : {p}")
    print(f"Resposta : {gerar_resposta(model_base, p)}")

Calculando métricas do modelo base (sem SFT)...

=== ANTES do SFT ===
  Entropia Cruzada : 3.3151
  Perplexidade     : 27.53
  Acurácia Tokens  : 41.37%

--- Respostas ANTES do SFT ---

Pergunta : Quem é o professor responsável pela disciplina de Tópicos em IA?
Resposta : O professor responsável pela disciplina de Tópicos em IA é [Professor X]

Pergunta : Qual é a titulação do professor X do Departamento de Computação?
Resposta : O professor X do Departamento de Computação possui o título de Mestrado em Informática.

Pergunta : Em que área de pesquisa atua o professor Y?
Resposta : O professor Y trabalha em área de Investigação e Pesquisa na Universidade Federal do Paraná (UFPR).


## Célula 9 — Dataset de Treino e SFT com Trainer

In [9]:
from transformers import TrainerCallback
import time

class SFTDataset(Dataset):
    def __init__(self, textos):
        self.enc = tokenizer(
            textos, return_tensors="pt",
            max_length=MAX_LEN, truncation=True, padding="max_length"
        )

    def __len__(self):
        return self.enc["input_ids"].shape[0]

    def __getitem__(self, idx):
        return {
            "input_ids":      self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx]
        }

print("Tokenizando dados de treino...")
train_ds = SFTDataset(textos_treino)
print("Tokenizando dados de avaliação...")
eval_ds  = SFTDataset(textos_eval)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(f"Treino: {len(train_ds):,} | Avaliação: {len(eval_ds):,}")

# ── Copia o modelo base para não sobrescrever os pesos originais ───────────
model_ft = copy.deepcopy(model_base)
model_ft.to(DEVICE)

OUTPUT_DIR = "./qwen25_sft_docentes"

# ── Callback para mostrar progresso em tempo real, passo a passo ───────────
class ProgressoTempoReal(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.inicio = time.time()
        total_passos = state.max_steps
        print(f"Treino iniciado — {total_passos} passos totais\n")

    def on_step_end(self, args, state, control, **kwargs):
        passo  = state.global_step
        total  = state.max_steps
        decorrido = time.time() - self.inicio
        passos_por_seg = passo / decorrido if decorrido > 0 else 0
        restante = (total - passo) / passos_por_seg if passos_por_seg > 0 else 0

        pct = (passo / total) * 100
        print(
            f"\r[{passo:>4}/{total}] {pct:5.1f}%  "
            f"| decorrido: {decorrido/60:5.1f}min  "
            f"| restante: ~{restante/60:5.1f}min",
            end="", flush=True
        )

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            print(f"\n  passo {state.global_step}: loss = {logs['loss']:.4f}")

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"\n>>> Época {int(state.epoch)} concluída <<<\n")

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,     # batch efetivo = 16
    gradient_checkpointing=True,      # economiza bastante VRAM

    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    fp16=(DEVICE.type == "cuda"),
    dataloader_num_workers=0,

    logging_steps=5,            # ← loss aparece a cada 5 passos, não 50
    logging_first_step=True,    # ← garante log já no passo 1
    disable_tqdm=False,
    report_to="none"
)

trainer = Trainer(
    model=model_ft,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
    callbacks=[ProgressoTempoReal()],   # ← callback de progresso
)

# ── Confirma se está rodando na GPU antes de começar ────────────────────────
print("CUDA disponível:", torch.cuda.is_available())
print("Device do modelo:", next(model_ft.parameters()).device)
print()

print(f"Iniciando SFT — {EPOCHS} épocas em {DEVICE}...\n")
resultado = trainer.train()

print("\n\nTreinamento concluído!")
print(f"  Loss final de treino : {resultado.training_loss:.4f}")
print(f"  Tempo total          : {resultado.metrics['train_runtime']:.0f}s")

Tokenizando dados de treino...
Tokenizando dados de avaliação...
Treino: 2,800 | Avaliação: 311
CUDA disponível: True
Device do modelo: cuda:0

Iniciando SFT — 3 épocas em cuda...

Treino iniciado — 525 passos totais



  0%|          | 0/525 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


[   1/525]   0.2%  | decorrido:   0.9min  | restante: ~449.1min

  0%|          | 1/525 [00:54<7:29:00, 51.41s/it]


  passo 1: loss = 3.5084
{'loss': 3.5084, 'grad_norm': 34.102081298828125, 'learning_rate': 1.8518518518518519e-06, 'epoch': 0.01}
[   2/525]   0.4%  | decorrido:   2.0min  | restante: ~535.5min

...

[   5/525]   1.0%  | decorrido:   5.8min  | restante: ~600.7min

  1%|          | 5/525 [05:50<10:26:51, 72.33s/it]


  passo 5: loss = 3.0360
{'loss': 3.036, 'grad_norm': 18.83498764038086, 'learning_rate': 9.259259259259259e-06, 'epoch': 0.03}
[   6/525]   1.1%  | decorrido:   7.1min  | restante: ~609.9min

  1%|          | 6/525 [07:03<10:37:52, 73.74s/it]

KeyboardInterrupt: 

## Célula 10 — Avaliação DEPOIS do SFT

In [ ]:
print("Calculando métricas do modelo após SFT...")
metricas_depois = calcular_metricas(model_ft, textos_eval)

print("\n=== DEPOIS do SFT ===")
print(f"  Entropia Cruzada : {metricas_depois['entropia_cruzada']}")
print(f"  Perplexidade     : {metricas_depois['perplexidade']}")
print(f"  Acurácia Tokens  : {metricas_depois['acuracia_tokens']}%")

print("\n--- Respostas DEPOIS do SFT ---")
for p in perguntas_teste:
    print(f"\nPergunta : {p}")
    print(f"Resposta : {gerar_resposta(model_ft, p)}")

## Célula 11 — Comparação Final ANTES vs. DEPOIS

In [ ]:
print("=" * 62)
print("        COMPARAÇÃO: ANTES vs. DEPOIS DO SFT")
print("=" * 62)
print(f"{'Métrica':<28} {'Antes':>10} {'Depois':>10} {'Δ':>10}")
print("-" * 62)

d_ec  = metricas_depois["entropia_cruzada"] - metricas_antes["entropia_cruzada"]
d_ppl = metricas_depois["perplexidade"]     - metricas_antes["perplexidade"]
d_acc = metricas_depois["acuracia_tokens"]  - metricas_antes["acuracia_tokens"]

print(f"{'Entropia Cruzada':<28} {metricas_antes['entropia_cruzada']:>10.4f} {metricas_depois['entropia_cruzada']:>10.4f} {d_ec:>+10.4f}")
print(f"{'Perplexidade (PPL)':<28} {metricas_antes['perplexidade']:>10.2f} {metricas_depois['perplexidade']:>10.2f} {d_ppl:>+10.2f}")
print(f"{'Acurácia de Tokens (%)':<28} {metricas_antes['acuracia_tokens']:>10.2f} {metricas_depois['acuracia_tokens']:>10.2f} {d_acc:>+10.2f}")
print("=" * 62)

var_ppl = (d_ppl / metricas_antes["perplexidade"]) * 100
print(f"\nVariação relativa da Perplexidade: {var_ppl:+.1f}%")
if d_ppl < 0:
    print("✓ Perplexidade REDUZIU — modelo melhorou no domínio docentesDC.")
else:
    print("✗ Perplexidade aumentou — revisar hiperparâmetros ou dados de treino.")

print("\n--- Comparação lado a lado das respostas geradas ---")
for p in perguntas_teste:
    print(f"\nPergunta : {p}")
    print(f"  ANTES  : {gerar_resposta(model_base, p)}")
    print(f"  DEPOIS : {gerar_resposta(model_ft, p)}")

## Célula 12 — Salvar Modelo e Resultados

In [ ]:
model_ft.save_pretrained("./qwen25_sft_docentes_final")
tokenizer.save_pretrained("./qwen25_sft_docentes_final")
print("Modelo salvo em ./qwen25_sft_docentes_final")

resultados = {
    "modelo": MODEL_NAME,
    "dataset": "docentesDC",
    "n_treino": len(textos_treino),
    "n_avaliacao": len(textos_eval),
    "hiperparametros": {
        "epocas": args.num_train_epochs,
        "batch_efetivo": args.per_device_train_batch_size * args.gradient_accumulation_steps,
        "learning_rate": args.learning_rate,
        "max_length": MAX_LEN
    },
    "metricas_antes": metricas_antes,
    "metricas_depois": metricas_depois,
    "delta": {
        "entropia_cruzada": round(d_ec, 4),
        "perplexidade": round(d_ppl, 2),
        "acuracia_tokens_pp": round(d_acc, 2)
    }
}

with open("resultados_sft_q2.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
print("Métricas salvas em resultados_sft_q2.json")

print("\n=== RESUMO FINAL ===")
print(f"Modelo base -> PPL: {metricas_antes['perplexidade']} | EC: {metricas_antes['entropia_cruzada']} | Acc: {metricas_antes['acuracia_tokens']}%")
print(f"Modelo SFT  -> PPL: {metricas_depois['perplexidade']} | EC: {metricas_depois['entropia_cruzada']} | Acc: {metricas_depois['acuracia_tokens']}%")